# Lag CATEs & Impact Persistence — DoubleML PLR
**Ashley Thompson — Capstone**

Two complementary analyses:

**Part 1 — Lagged sentiment → current return** *(reverse causality check)*  
Do old tweets still move today's price? Tests the no-reversal condition (Gu & Kurov 2020).
Only lag 1 should be significant; persistence at lags 3–7 signals reverse causality.

**Part 2 — Current sentiment → future return** *(research question)*  
How long does today's tweet impact last? Hypothesis: Twitter effects decay faster than news effects.

> **Runtime → Change runtime type → T4 GPU** before running. This notebook is compute-intensive — expect a long runtime.

## 1. Install dependencies

In [ ]:
!pip install -q doubleml xgboost scikit-learn

## 2. Load data
Run Option A or Option B, then skip the other.

In [ ]:
# Option A — Google Drive (data only)
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Capstone/panel_long.csv'

In [ ]:
# Option B — Manual upload (data only)
from google.colab import files
uploaded = files.upload()
DATA_PATH = 'panel_long.csv'

In [ ]:
# GitHub output setup — run after whichever data option above
from google.colab import userdata
import os, subprocess

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR     = '/content/Capstone'
OUTPUT_PATH  = REPO_DIR + '/output/'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone',
         f'https://{GITHUB_TOKEN}@github.com/ATHOMP-03/Capstone.git',
         REPO_DIR],
        check=True,
    )
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'Output path ready: {OUTPUT_PATH}')

## 3. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import doubleml as dml
from xgboost import XGBRegressor

np.random.seed(42)
os.makedirs(OUTPUT_PATH, exist_ok=True)

long = pd.read_csv(DATA_PATH, parse_dates=['date'])
long = long.sort_values(['ticker', 'date']).reset_index(drop=True)
print(f'Loaded {len(long):,} rows x {long.shape[1]} cols')
long.head()

In [ ]:
def make_xgb():
    return XGBRegressor(
        n_estimators     = 500,
        learning_rate    = 0.05,
        max_depth        = 6,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        device           = 'cuda',
        tree_method      = 'hist',
        random_state     = 42,
        n_jobs           = -1,
    )

LAGS = [1, 2, 3, 5, 7]

# Excludes current twitter_sent and news_sent — both treated as treatments
# at different lags; including current values would introduce endogeneity
BASE_CONFOUNDERS = [
    'px_high', 'px_low', 'mkt_cap', 'total_equity', 'debt_to_equity',
    'volume', 'twitter_count', 'rsi_30', 'ma_50',
    'twitter_neg_count', 'lag1', 'lag2', 'lag3', 'lag5', 'lag7',
]

## 4. Part 1 — Lagged sentiment → current return
*(Reverse causality check — univariate)*

Each lag run separately. Results feed into the no-reversal test below.

In [ ]:
# Create lagged sentiment variables within each ticker
for lag_n in LAGS:
    long[f'twitter_sent_lag{lag_n}'] = long.groupby('ticker')['twitter_sent'].shift(lag_n)
    long[f'news_sent_lag{lag_n}']    = long.groupby('ticker')['news_sent'].shift(lag_n)

results = []

for lag_n in LAGS:
    for treatment, label in [
        (f'twitter_sent_lag{lag_n}', 'twitter_sent'),
        (f'news_sent_lag{lag_n}',    'news_sent'),
    ]:
        df = (
            long[['return', treatment] + BASE_CONFOUNDERS]
            .dropna()
            .reset_index(drop=True)
        )
        model = dml.DoubleMLPLR(
            dml.DoubleMLData(df, y_col='return', d_cols=treatment, x_cols=BASE_CONFOUNDERS),
            ml_l=make_xgb(), ml_m=make_xgb(), n_folds=5, n_rep=20,
        )
        model.fit()
        model.bootstrap(method='normal', n_rep_boot=1000)

        coef     = model.summary['coef'].iloc[0]
        pval     = model.summary['P>|t|'].iloc[0]
        ci_lower = model.confint(level=0.95)['2.5 %'].iloc[0]
        ci_upper = model.confint(level=0.95)['97.5 %'].iloc[0]

        print(f'lag={lag_n} | {label}: coef={coef:.6f}  p={pval:.4f}  CI=[{ci_lower:.6f}, {ci_upper:.6f}]')
        results.append({'treatment': label, 'lag': lag_n, 'coef': coef, 'p_value': pval, 'ci_lower': ci_lower, 'ci_upper': ci_upper})

results_df = pd.DataFrame(results)
print('\nLAG DECAY TABLE (univariate)')
print(results_df.sort_values(['lag','treatment']).to_string(index=False))

## 5. No-reversal test — simultaneous specification (Gu & Kurov 2020)
All sentiment lags estimated jointly so each coefficient is conditioned on the others.

**Pass:** lag 1 significant, lags 2–7 insignificant → permanent information effect  
**Fail:** lags 3–7 persist → momentum or reverse causality

In [ ]:
for channel, label in [('twitter_sent', 'Twitter sentiment'), ('news_sent', 'News sentiment')]:
    lag_cols = [f'{channel}_lag{n}' for n in LAGS]
    df_nr = long[['return'] + lag_cols + BASE_CONFOUNDERS].dropna().reset_index(drop=True)

    nr_model = dml.DoubleMLPLR(
        dml.DoubleMLData(df_nr, y_col='return', d_cols=lag_cols, x_cols=BASE_CONFOUNDERS),
        ml_l=make_xgb(), ml_m=make_xgb(), n_folds=5, n_rep=20,
    )
    nr_model.fit()
    nr_model.bootstrap(method='normal', n_rep_boot=1000)

    print(f'\n--- {label} ---')
    verdicts = {}
    for col in lag_cols:
        coef     = nr_model.summary.loc[col, 'coef']
        pval     = nr_model.summary.loc[col, 'P>|t|']
        ci_lower = nr_model.confint(level=0.95).loc[col, '2.5 %']
        ci_upper = nr_model.confint(level=0.95).loc[col, '97.5 %']
        sig = '*' if pval < 0.05 else ''
        verdicts[col] = pval < 0.05
        print(f'  {col:<22} coef={coef:>9.6f}  p={pval:.4f}  CI=[{ci_lower:.6f}, {ci_upper:.6f}]  {sig}')

    lag1_col  = f'{channel}_lag1'
    later_sig = [c for c in lag_cols if c != lag1_col and verdicts[c]]
    print(f'\n  VERDICT:')
    if verdicts[lag1_col] and not later_sig:
        print('  PASS — lag 1 significant, lags 2-7 insignificant. Consistent with permanent information effect.')
    elif not verdicts[lag1_col]:
        print('  INCONCLUSIVE — lag 1 not significant.')
    else:
        print(f'  FAIL — significant persistence at: {", ".join(later_sig)}. Suggests momentum or reverse causality.')

## 6. Part 2 — Current sentiment → future return
*(Impact persistence — research question)*

Creates lead variables of return and regresses today's sentiment on each future horizon.

**Hypothesis:** `twitter_sent` effect decays by lead 2–3; `news_sent` persists to lead 5–7.

In [ ]:
# Create lead return variables within each ticker
for lead_n in LAGS:
    long[f'return_lead{lead_n}'] = long.groupby('ticker')['return'].shift(-lead_n)

lead_results = []

for lead_n in LAGS:
    outcome = f'return_lead{lead_n}'
    for treatment, label in [('twitter_sent', 'twitter_sent'), ('news_sent', 'news_sent')]:
        df = (
            long[[outcome, treatment] + BASE_CONFOUNDERS]
            .dropna()
            .reset_index(drop=True)
        )
        model = dml.DoubleMLPLR(
            dml.DoubleMLData(df, y_col=outcome, d_cols=treatment, x_cols=BASE_CONFOUNDERS),
            ml_l=make_xgb(), ml_m=make_xgb(), n_folds=5, n_rep=20,
        )
        model.fit()
        model.bootstrap(method='normal', n_rep_boot=1000)

        coef     = model.summary['coef'].iloc[0]
        pval     = model.summary['P>|t|'].iloc[0]
        ci_lower = model.confint(level=0.95)['2.5 %'].iloc[0]
        ci_upper = model.confint(level=0.95)['97.5 %'].iloc[0]

        print(f'lead={lead_n} | {label}: coef={coef:.6f}  p={pval:.4f}  CI=[{ci_lower:.6f}, {ci_upper:.6f}]')
        lead_results.append({'treatment': label, 'lead': lead_n, 'coef': coef, 'p_value': pval, 'ci_lower': ci_lower, 'ci_upper': ci_upper})

lead_df = pd.DataFrame(lead_results)
print('\nIMPACT PERSISTENCE TABLE')
print(lead_df.sort_values(['lead','treatment']).to_string(index=False))
print('\n* If twitter_sent insignificant by lead 2-3 but news_sent persists to lead 5-7,')
print('  tweet cycles are shorter than news cycles — consistent with the core hypothesis.')

## 7. Export — Publication-ready LaTeX tables
Saves two `.tex` files to your output folder:
- `lag_decay_table.tex` — univariate lag decay (Part 1)
- `impact_persistence_table.tex` — current sentiment → future return (Part 2)

In [ ]:
def results_df_to_latex(df, period_col, title, notes=''):
    """Convert a lag/lead results DataFrame to a publication-ready LaTeX table."""
    def stars(p):
        if p < 0.01: return '***'
        if p < 0.05: return '**'
        if p < 0.1:  return '*'
        return ''

    lines = [
        r'\begin{table}[htbp]',
        r'\centering',
        f'\\caption{{{title}}}',
        r'\begin{tabular}{llcccc}',
        r'\hline\hline',
        f'Treatment & {period_col.title()} & Coefficient & Std Err & p-value & 95\\% CI \\\\',
        r'\hline',
    ]
    for _, r in df.iterrows():
        sig       = stars(r['p_value'])
        treat_lbl = str(r['treatment']).replace('_', r'\_')
        coef_str  = f"{r['coef']:.6f}{sig}"
        se_str    = f"{r.get('se', '')}"
        pval_str  = f"{r['p_value']:.4f}"
        ci_str    = f"[{r['ci_lower']:.6f}, {r['ci_upper']:.6f}]"
        lines.append(f"{treat_lbl} & {int(r[period_col])} & {coef_str} & {se_str} & {pval_str} & {ci_str} \\\\")
    lines += [
        r'\hline',
        'Estimator & \\multicolumn{5}{l}{DoubleML PLR, XGBoost (GPU), 5-fold, 20 reps} \\\\',
        r'\hline\hline',
    ]
    if notes:
        lines.append(f'\\multicolumn{{6}}{{l}}{{\\footnotesize \\textit{{Notes:}} {notes}}} \\\\')
    lines += [r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)

In [ ]:
# Lag decay table (Part 1 — univariate)
lag_tex = results_df_to_latex(
    results_df.sort_values(['lag', 'treatment']),
    period_col='lag',
    title='Lag Decay --- Lagged Sentiment and Current Return (DoubleML PLR)',
    notes=(
        'Each lag estimated separately (univariate). '
        'Outcome: current-day return. '
        'BASE\\_CONFOUNDERS excludes contemporaneous sentiment to avoid endogeneity. '
        'XGBoost learner, 5-fold, 20 reps, 1000 bootstrap draws. '
        '*** p$<$0.01, ** p$<$0.05, * p$<$0.1.'
    ),
)
with open(OUTPUT_PATH + 'lag_decay_table.tex', 'w') as f:
    f.write(lag_tex)
print('Saved lag_decay_table.tex')

# Impact persistence table (Part 2 — lead returns)
lead_tex = results_df_to_latex(
    lead_df.sort_values(['lead', 'treatment']),
    period_col='lead',
    title='Impact Persistence --- Current Sentiment and Future Return (DoubleML PLR)',
    notes=(
        'Each lead estimated separately. '
        'Outcome: return\\_lead$n$ (return $n$ days ahead). '
        'Hypothesis: twitter\\_sent decays by lead 2--3; news\\_sent persists to lead 5--7 (Gu \\& Kurov 2020). '
        'XGBoost learner, 5-fold, 20 reps, 1000 bootstrap draws. '
        '*** p$<$0.01, ** p$<$0.05, * p$<$0.1.'
    ),
)
with open(OUTPUT_PATH + 'impact_persistence_table.tex', 'w') as f:
    f.write(lead_tex)
print('Saved impact_persistence_table.tex')

In [ ]:
# Push outputs to GitHub
import os, subprocess

os.chdir(REPO_DIR)
subprocess.run(['git', 'config', 'user.email', 'ATHOMP-03@users.noreply.github.com'])
subprocess.run(['git', 'config', 'user.name',  'ATHOMP-03'])
subprocess.run(['git', 'add', 'output/'])

staged   = subprocess.run(['git', 'diff', '--cached', '--quiet'])
unpushed = subprocess.run(
    ['git', 'log', 'origin/main..HEAD', '--oneline'],
    capture_output=True, text=True
)

if staged.returncode != 0:
    subprocess.run(['git', 'commit', '-m', 'Add lag CATEs results from Colab'], check=True)

if staged.returncode != 0 or unpushed.stdout.strip():
    subprocess.run(['git', 'push'], check=True)
    print('Pushed to GitHub.')
else:
    print('Nothing new to commit or push.')